# Genetic Algorithm
Genetic representation: <br>
List of clients in order of being calculated.

Mutation: <br>
Swap mutation to change the path randomly. Each mutation will use one parent to make sure no duplicate clients get added.

Tournament selection: <br>
Used to get a fair spread of good and bad states.

Elitism survivor selection: <br>
To make sure we do not go back in progress.

In [32]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pyarrow
import pandas as pd
import osmnx as ox

In [ ]:
# Initialize all necessary parts of the genetic algorithm
import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

POPULATION_SIZE = 200
EPOCHS = 500

MUTATION_RATE = 0.20
MUTATION_STRIDE = 2

TOURNAMENT_SIZE = 5
ELITE_SIZE = max(1, int(0.10 * POPULATION_SIZE))

TRAVEL_SPEED_KMH = 40
MAX_EMPLOYEE_WORK_TIME_MIN = 8 * 60

print("GA initialized:")
print(f"  seed={RANDOM_SEED}, population={POPULATION_SIZE}, epochs={EPOCHS}")
print(f"  mutation_rate={MUTATION_RATE}, mutation_stride={MUTATION_STRIDE}")
print(f"  tournament={TOURNAMENT_SIZE}, elite_size={ELITE_SIZE}")
print(f"  speed={TRAVEL_SPEED_KMH} km/h, max_work={MAX_EMPLOYEE_WORK_TIME_MIN} min")


In [9]:
# ============= Mutation Function =============
def swap_mutation(parent):
    """
    Swap mutation: takes a single parent (list of client indices),
    creates a mutant by swapping MUTATION_STRIDE random position pairs.
    
    Args:
        parent: list of client indices representing a route order
        
    Returns:
        mutant: mutated copy of parent with swapped positions
    """
    mutant = parent.copy()
    
    # Perform MUTATION_STRIDE swaps
    for _ in range(MUTATION_STRIDE):
        # Select two random distinct indices
        i, j = np.random.choice(len(mutant), size=2, replace=False)
        # Swap them
        mutant[i], mutant[j] = mutant[j], mutant[i]
    
    return mutant


# Test the mutation function
if __name__ == "__main__":
    # Example: create a parent route with 10 clients
    test_parent = list(range(10))
    print(f"Original parent: {test_parent}")
    print(f"Mutated child:   {swap_mutation(test_parent)}")
    print(f"Mutated child:   {swap_mutation(test_parent)}")
    print(f"Mutated child:   {swap_mutation(test_parent)}")

Original parent: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Mutated child:   [8, 0, 2, 3, 4, 5, 6, 7, 1, 9]
Mutated child:   [0, 7, 9, 3, 4, 5, 6, 1, 8, 2]
Mutated child:   [0, 5, 8, 3, 4, 1, 6, 7, 2, 9]


In [14]:
# ============= Elitism Selection =============
def elitism_selection(population, fitness_scores):
    """
    Elitism selection: preserve the top ELITE_SIZE individuals from the population
    based on their fitness scores (lower fitness is better for minimization).
    
    Args:
        population: list of individuals (each individual is a list of client indices)
        fitness_scores: list of fitness values corresponding to each individual
        
    Returns:
        elite_individuals: list of the ELITE_SIZE best individuals
        elite_fitness: list of fitness scores for the elite individuals
    """
    # Get indices sorted by fitness (ascending - lower is better)
    sorted_indices = np.argsort(fitness_scores)[:ELITE_SIZE]
    
    # Extract elite individuals and their fitness scores
    elite_individuals = [population[i] for i in sorted_indices]
    elite_fitness = [fitness_scores[i] for i in sorted_indices]
    
    return elite_individuals, elite_fitness


# Test elitism selection
if __name__ == "__main__":
    # Example population and fitness scores
    test_population = [
        [0, 1, 2, 3, 4],
        [4, 3, 2, 1, 0],
        [1, 2, 3, 4, 0],
        [2, 0, 4, 1, 3],
        [3, 4, 1, 0, 2],
        [0, 4, 2, 3, 1],
    ]
    test_fitness = [100, 80, 120, 95, 110, 75]  # Lower is better
    
    elite_pop, elite_fits = elitism_selection(test_population, test_fitness)
    
    print(f"Original fitness scores: {test_fitness}")
    print(f"\nElite individuals (top {ELITE_SIZE}):")
    for i, (ind, fit) in enumerate(zip(elite_pop, elite_fits)):
        print(f"  {i+1}. Fitness: {fit}, Route: {ind}")


# ============= Initial Population =============
NUM_CLIENTS = 100
client_indices = list(range(NUM_CLIENTS))
initial_population = [np.random.permutation(client_indices).tolist() for _ in range(POPULATION_SIZE)]

print(f"\nInitialized population with {len(initial_population)} individuals.")
print(f"Each individual contains {len(initial_population[0])} client indices.")
print("First individual:")
print(initial_population[0])

Original fitness scores: [100, 80, 120, 95, 110, 75]

Elite individuals (top 5):
  1. Fitness: 75, Route: [0, 4, 2, 3, 1]
  2. Fitness: 80, Route: [4, 3, 2, 1, 0]
  3. Fitness: 95, Route: [2, 0, 4, 1, 3]
  4. Fitness: 100, Route: [0, 1, 2, 3, 4]
  5. Fitness: 110, Route: [3, 4, 1, 0, 2]

Initialized population with 100 individuals.
Each individual contains 100 client indices.
First individual:
[55, 4, 79, 94, 35, 49, 68, 74, 25, 44, 52, 24, 66, 56, 72, 93, 91, 57, 81, 15, 0, 75, 21, 32, 7, 77, 88, 42, 53, 8, 34, 3, 50, 20, 45, 2, 83, 29, 76, 98, 41, 47, 69, 6, 96, 89, 19, 60, 61, 90, 11, 39, 5, 48, 38, 37, 46, 17, 28, 36, 70, 43, 30, 16, 84, 27, 51, 59, 92, 40, 9, 97, 58, 10, 99, 18, 65, 12, 78, 31, 63, 80, 85, 87, 64, 62, 13, 67, 22, 95, 86, 73, 26, 1, 33, 71, 82, 14, 54, 23]


In [31]:
clients_with_coords = pd.read_csv("../output/clients_with_coords.csv")
heerlen_edge_table = pd.read_csv("../output/heerlen_edge_table.csv")



for i in range(len(initial_population[0])):
    client_index = initial_population[0][i]
    client_info = clients_with_coords.iloc[client_index]
    coords = (client_info['longitude'], client_info['latitude'])


for i in range(len(heerlen_edge_table)):
    edge_info = heerlen_edge_table.iloc[i]
    geometry = str(edge_info['geometry'])
    if geometry.upper().startswith('LINESTRING'):
        start = geometry.find('(')
        end = geometry.rfind(')')
        geometry = geometry[start + 1:end] if start != -1 and end != -1 else geometry
    print(geometry)


6.021233 50.821986, 6.0221654 50.821062, 6.022496 50.8207246, 6.0225571 50.8206624, 6.0226346 50.8205837
6.0223386 50.8214182, 6.022286 50.8214464, 6.0219269 50.8216392
6.0207387 50.8222699, 6.0208651 50.8221973, 6.021233 50.821986
6.0210875 50.8224672, 6.0187792 50.824743
6.0210875 50.8224672, 6.0209906 50.8227166, 6.020955 50.822828, 6.0209356 50.8229132, 6.0209277 50.8229847
6.0198126 50.8243229, 6.0196696 50.8243687, 6.0193777 50.8244719, 6.0192571 50.8245188, 6.0192045 50.8245393, 6.0187792 50.824743
6.0184986 50.8246761, 6.0198755 50.8233256, 6.0206071 50.822604, 6.0209732 50.8222439, 6.021233 50.821986
6.0184986 50.8246761, 6.0185763 50.8244589, 6.0186343 50.8242967
6.0187792 50.824743, 6.0186237 50.824898, 6.018347 50.825164, 6.0149201 50.8285514
6.0149201 50.8285514, 6.0115205 50.8318882, 6.0088705 50.834497, 6.007052 50.836296, 6.0062098 50.8370244, 6.005268 50.8377458, 6.004357 50.838398, 6.00313 50.8391546, 6.002824 50.8393367, 6.0022581 50.8396507, 6.001101 50.8402432
6.01